# bioseq bench `report.json` viewer

Self-contained, single-file notebook for inspecting a `report.json`
produced by `./build.sh bench`. The bench writes the file to
`build/report.json` (or whatever path was passed via the
`BIOSEQ_BENCH_REPORT_PATH` environment variable). Override the
`REPORT_PATH` constant in the next cell to point somewhere else.

No external dependencies beyond `plotly` (already available on this
machine) and the Python standard library. To run:

1. Open in Jupyter (`jupyter notebook view_report.ipynb`) or VS Code.
2. Execute the cells top-to-bottom. The last cell renders every
   chart in the report.

Schema: `bioseq-bench-report-v1` — see
`main/bioseq/RESEARCH/BENCHMARKING_SPEEDUP/REPORT_PLOTLY_JSON_DATA.md`
for what each chart means.

In [ ]:
# ----- configuration -----
# Either an absolute path or a path relative to the notebook directory.
# The bench writes to ./report.json by default; the build script invokes
# the binary from main/build/, so the canonical location is
# main/build/report.json.
REPORT_PATH = '../../../build/report.json'

import json
import os
import sys

import plotly.io
from IPython.display import display, Markdown

with open(REPORT_PATH, 'r') as report_file:
    report = json.load(report_file)

expected_schema = 'bioseq-bench-report-v1'
actual_schema = report.get('schema', '<missing>')
if actual_schema != expected_schema:
    print(
        f'WARNING: schema mismatch — expected {expected_schema!r}, '
        f'got {actual_schema!r}. The notebook may render incorrectly.',
        file=sys.stderr,
    )

print(f'Loaded {os.path.abspath(REPORT_PATH)}')
print(f'  schema:           {actual_schema}')
print(f'  sample_count:     {report.get("sample_count", "?")}')
print(f'  algorithm_count:  {report.get("algorithm_count", "?")}')
print(f'  chart_count:      {len(report.get("charts", {}))}')

In [ ]:
# Render every chart, in the order they appear in the report.
# Each chart prints its short_description as a heading and its
# long_description as a paragraph below the figure. Figures use
# Plotly's interactive renderer — hover for tooltips, drag to zoom,
# double-click to reset.

for chart_id, chart in report.get('charts', {}).items():
    short_description = chart.get('short_description', chart_id)
    long_description = chart.get('long_description', '')
    plotly_payload = chart.get('plotly')
    if plotly_payload is None:
        display(Markdown(f'### `{chart_id}` — _no plotly payload, skipped_'))
        continue
    display(Markdown(f'## `{chart_id}` — {short_description}'))
    try:
        figure = plotly.io.from_json(json.dumps(plotly_payload))
    except Exception as exc:  # noqa: BLE001 — surface any parse error inline
        display(Markdown(
            f'**plotly.io.from_json failed:** `{exc}`'))
        continue
    figure.show()
    if long_description:
        display(Markdown(f'> {long_description}'))